# Session 05 — Evaluating Generative Models · Guided lab

*Desarrollo y Evaluación de Modelos Propios · Universidad del Rosario · Rosario GSB*

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/05-evaluating-generative-models/lab.ipynb)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/05-evaluating-generative-models/lab.ipynb)

Today you measure the quality of four small models with the metrics from the slides, and you
learn to read those numbers critically: what each metric rewards, what it misses, and how much
it moves when the data changes.

**Learning objectives**
1. Compute the perplexity of a language model and explain why it depends on the tokenizer (RAE 2).
2. Score machine translations with BLEU and chrF using `sacrebleu`, and inspect the worst cases (RAE 2).
3. Score summaries with ROUGE and embedding similarity, always against a simple baseline (RAE 2).
4. Estimate a classifier's performance with stratified k-fold cross-validation and analyse its confusions (RAE 2).

**Time plan (60 min)**

| Part | Topic | Time |
|---|---|---|
| 0 | Setup | 3' |
| 1 | Perplexity with GPT-2: news, tweets and Spanish | 12' |
| 2 | EN→ES translation: BLEU and chrF | 12' |
| 3 | Summarization: ROUGE, a baseline and semantic similarity | 15' |
| 4 | Classification: stratified k-fold, confusion matrix and error analysis | 15' |
| — | Wrap-up | 3' |

**Kaggle checklist**
- Settings → **Internet on** (models and datasets are downloaded from the Hugging Face Hub).
- Settings → Accelerator: **none** is enough today (everything runs on CPU in a few minutes). A GPU only makes Parts 2–3 a little faster.
- Run the cells in order. If you get lost, *Run → Restart & run all*.

In [ ]:
# --- Course setup (do not edit) ---
import os, sys, random, subprocess, importlib
FAST_DEV_RUN = os.environ.get("COURSE_FAST_DEV_RUN") == "1"  # used by the course smoke test
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules

def ensure(package, import_name=None):
    """Install a package only if it is missing (Kaggle already ships most of them)."""
    try:
        importlib.import_module(import_name or package)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import socket
if os.environ.get("HF_HUB_OFFLINE") != "1":  # models and datasets come from the Hugging Face Hub
    try:
        socket.create_connection(("huggingface.co", 443), timeout=10).close()
    except OSError:
        raise RuntimeError(
            "No internet connection: this notebook downloads models and datasets from Hugging Face.\n"
            "Kaggle: in the right-hand panel open Settings (Session options) and switch Internet on. "
            "The switch only appears once your account is phone-verified (kaggle.com/settings). "
            "Then run this cell again. Colab has internet by default. "
            "On your own computer: check your connection, or set HF_HUB_OFFLINE=1 if the models are already cached."
        ) from None

import numpy as np
import torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | Kaggle: {IN_KAGGLE} | Colab: {IN_COLAB} | Fast dev run: {FAST_DEV_RUN}")
# --- end of course setup ---
ensure("sacrebleu")
ensure("rouge-score", "rouge_score")
ensure("sentence-transformers", "sentence_transformers")
ensure("sentencepiece")  # needed by the MarianMT (opus-mt) tokenizer

Imports and the sizes used in this notebook. `FAST_DEV_RUN` is only for the course's automatic
test: it runs the same code on tiny samples. You do not need to change anything here.

In [ ]:
import gc, math, re, time, warnings
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore", message=".*sacremoses.*")
pd.set_option("display.max_colwidth", 120)

N_TRANSLATIONS = 5 if FAST_DEV_RUN else 60     # Part 2
MT_MAX_NEW_TOKENS = 64 if FAST_DEV_RUN else 128
N_SUMMARIES = 3 if FAST_DEV_RUN else 12        # Part 3
SUM_MAX_NEW_TOKENS = 32 if FAST_DEV_RUN else 64
NUM_BEAMS = 1 if FAST_DEV_RUN else 4
N_PER_INTENT = 40 if FAST_DEV_RUN else None    # Part 4 (None = all training examples)
N_HOLDOUT_REPEATS = 3 if FAST_DEV_RUN else 20

def release_memory():
    """Free RAM after `del model` (Kaggle CPU sessions have limited memory)."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Ready.")

## Part 1 — Perplexity: how surprised is the model?

A causal language model assigns a probability to every next token. **Perplexity** is the
exponential of the average negative log-likelihood (NLL) over the $N$ tokens of a text:

$$\text{PPL} = \exp\Big(-\frac{1}{N}\sum_{t=1}^{N} \log p(x_t \mid x_{<t})\Big)$$

A perplexity of 20 means the model is, on average, as uncertain as if it had to choose
uniformly among 20 tokens. Lower is better **for the same text and the same tokenizer**.

Two practical details (both in the code below):
- We prepend GPT-2's special `<|endoftext|>` token so that the first real token is also scored.
- GPT-2 sees at most 1,024 tokens. For longer texts we slide a window with a `stride`, so that
  every token is scored once with as much left context as possible (short texts use one window).

In [ ]:
LM_NAME = "openai-community/gpt2"
gpt2_tok = AutoTokenizer.from_pretrained(LM_NAME)
gpt2 = AutoModelForCausalLM.from_pretrained(LM_NAME).to(DEVICE).eval()
print(f"{LM_NAME}: {sum(p.numel() for p in gpt2.parameters()) / 1e6:.0f}M parameters, "
      f"vocabulary of {len(gpt2_tok):,} tokens, context of {gpt2.config.n_positions} tokens")


def token_nlls(text, model, tokenizer, stride=512):
    """NLL (in nats) of every token of `text`, using a sliding window for long texts."""
    ids = tokenizer(tokenizer.bos_token + text, return_tensors="pt").input_ids.to(DEVICE)
    max_len = model.config.n_positions
    seq_len = ids.size(1)
    pieces, prev_end = [], 0
    for begin in range(0, seq_len, stride):
        end = min(begin + max_len, seq_len)
        window = ids[:, begin:end]
        with torch.no_grad():
            logits = model(window).logits.float()
        logprobs = torch.log_softmax(logits[0, :-1], dim=-1)            # predictions for positions begin+1 … end-1
        nll = -logprobs.gather(1, window[0, 1:].unsqueeze(1)).squeeze(1)
        new = end - max(prev_end, begin + 1)                            # tokens not scored by the previous window
        pieces.append(nll[-new:])
        prev_end = end
        if end == seq_len:
            break
    return torch.cat(pieces).cpu(), ids[0, 1:].cpu()                   # drop the BOS token from the ids


def perplexity_report(text, model=None, tokenizer=None):
    """Perplexity per token and per word (GPT-2 unless another model is given)."""
    nll, ids = token_nlls(text, model or gpt2, tokenizer or gpt2_tok)
    n_words = len(text.split())
    return {
        "tokens": len(ids),
        "words": n_words,
        "tokens/word": round(len(ids) / n_words, 2),
        "PPL per token": round(math.exp(nll.mean().item()), 1),
        "PPL per word": round(math.exp(nll.sum().item() / n_words), 1),
    }

Four short texts written for this lab (Andes Bank is a fictional company). The last one uses
exactly the same words as the news text, in a random order.

In [ ]:
TEXTS = {
    "news (EN)": ("Andes Bank reported on Tuesday that its mobile app handled more customer requests "
                  "than its branches during the third quarter. The bank said it will expand its digital "
                  "support team and open two new service centres next year. Analysts expect other "
                  "regional lenders to follow a similar strategy."),
    "tweet (EN)": ("ngl the new andes bank app update is 🔥 but why does it log me out every 5 mins?? "
                   "fix pls #fail #bankinglife"),
    "news (ES)": ("Andes Bank informó el martes que su aplicación móvil atendió más solicitudes de clientes "
                  "que sus sucursales durante el tercer trimestre. El banco anunció que ampliará su equipo "
                  "de soporte digital y abrirá dos nuevos centros de servicio el próximo año."),
}
_words = TEXTS["news (EN)"].split()
random.Random(SEED).shuffle(_words)
TEXTS["news (EN), shuffled"] = " ".join(_words)

ppl_table = pd.DataFrame({name: perplexity_report(t) for name, t in TEXTS.items()}).T
display(ppl_table)

ax = ppl_table["PPL per token"].plot.barh(figsize=(7, 2.8), color="#0F766E")
ax.set_xlabel("GPT-2 perplexity per token (lower = less surprised)")
ax.invert_yaxis()
plt.tight_layout(); plt.show()

**Reading the table**
- The shuffled text has the same words as the news text, yet a much higher perplexity: GPT-2 has
  learned word order, not just vocabulary. Perplexity measures *fluency under the model*.
- The tweet is short, informal and full of rare pieces (emoji, hashtags): GPT-2 is more surprised.
- Look at the **tokens/word** column for Spanish. GPT-2's tokenizer was built mostly from English
  web text, so each Spanish word is cut into about two pieces, and the model pays a price for
  every piece. Per token, the Spanish news looks a few times "harder" than the English news; per
  word (total NLL divided by the number of words), the gap is hundreds of times larger. Same model,
  same texts: only the unit changed, and the story changed with it.

That is why perplexities are only comparable **on the same text with the same tokenizer**. To
compare models with different tokenizers, normalise by a unit they share (words, characters or
bytes) or, better, compare them on the task you care about.

### Same tokenizer, two models: now the comparison is fair

`distilbert/distilgpt2` is a smaller model distilled from GPT-2 that uses **the same tokenizer**,
so its per-token perplexity on the same texts can be compared directly with GPT-2's.

In [ ]:
distil_tok = AutoTokenizer.from_pretrained("distilbert/distilgpt2")
distil = AutoModelForCausalLM.from_pretrained("distilbert/distilgpt2").to(DEVICE).eval()
assert distil_tok.get_vocab() == gpt2_tok.get_vocab(), "the two tokenizers should be identical"

compare = pd.DataFrame({
    "GPT-2 (124M)": {n: perplexity_report(t)["PPL per token"] for n, t in TEXTS.items()},
    "DistilGPT2 (82M)": {n: perplexity_report(t, distil, distil_tok)["PPL per token"] for n, t in TEXTS.items()},
})
display(compare)
del distil
release_memory()

### Which tokens surprised the model?

Perplexity is an average. Looking at the individual tokens tells you *where* the model struggles.
Surprisal in bits is $-\log_2 p(x_t \mid x_{<t})$: 1 bit is a coin flip, 10 bits is roughly a
1-in-1,000 guess.

In [ ]:
nll, ids = token_nlls(TEXTS["tweet (EN)"], gpt2, gpt2_tok)
surprisal = pd.DataFrame({
    "token": [repr(gpt2_tok.decode([int(i)])) for i in ids],
    "surprisal (bits)": (nll.double() / math.log(2)).numpy().round(1),
})
display(surprisal.sort_values("surprisal (bits)", ascending=False).head(8))
print("Note: an emoji is split into several byte-level tokens, which print as '�' one by one.")

✋ **Checkpoint (Zoom chat):**
1. Which text surprised GPT-2 the most, and why?
2. From the per-token numbers alone, would you conclude that GPT-2 is "good at Spanish"? What does the per-word column add?
3. A vendor says: "our model has perplexity 12, the competitor's has 19". What do you ask before believing it is better?

🧪 **Try it:** write one sentence from your own work domain and compute its report. Then change one
key word for a wrong one (e.g., "approved" → "banana") and compare.

<details><summary>Show a solution</summary>

```python
mine = "The client asked us to increase the credit limit on her card before the trip."
wrong = "The client asked us to increase the banana limit on her card before the trip."
display(pd.DataFrame({"original": perplexity_report(mine), "one wrong word": perplexity_report(wrong)}).T)
```
One unexpected word raises the perplexity of the whole sentence. Note that perplexity does not
check facts: a fluent false sentence can have a low perplexity.
</details>

In [ ]:
del gpt2
release_memory()
print("GPT-2 released.")

## Part 2 — Machine translation EN→ES: BLEU and chrF

We translate English sentences from `Helsinki-NLP/opus_books` (sentence-aligned public-domain
books) with `Helsinki-NLP/opus-mt-en-es`, a MarianMT model, and compare each output with the
human Spanish translation (the **reference**).

- **BLEU** (Papineni et al., 2002): clipped n-gram *precision* for n = 1…4, geometric mean, times a
  **brevity penalty** that punishes outputs shorter than the reference. Scale 0–100.
- **chrF** (Popović, 2015): F-score over *character* n-grams (β = 2, recall weighs more). Friendlier to
  languages with rich morphology such as Spanish (*aprobó / aprobado / aprobaron* share characters).

We use `sacrebleu`, the standard implementation, so that scores are reproducible (it prints a
*signature* with the exact settings).

In [ ]:
books = load_dataset("Helsinki-NLP/opus_books", "en-es", split="train")
print(books)
print(books[0]["translation"])   # the first rows are headers and boilerplate


def is_clean_pair(pair, min_words=5, max_words=40):
    """Skip headers, very short lines and very long sentences."""
    en, es = pair["en"].strip(), pair["es"].strip()
    return (min_words <= len(en.split()) <= max_words and len(es.split()) >= min_words
            and not en.isupper() and not en.lower().startswith(("source:", "chapter")))


pool = [i for i in range(100, len(books)) if is_clean_pair(books[i]["translation"])]
mt_idx = sorted(random.Random(SEED).sample(pool, N_TRANSLATIONS))
sources = [books[i]["translation"]["en"] for i in mt_idx]
references = [books[i]["translation"]["es"] for i in mt_idx]
print(f"{len(pool):,} clean pairs; we use {len(mt_idx)}.")

In [ ]:
MT_NAME = "Helsinki-NLP/opus-mt-en-es"
mt_tok = AutoTokenizer.from_pretrained(MT_NAME)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(MT_NAME).to(DEVICE).eval()


def translate(texts, batch_size=8):
    out = []
    for i in range(0, len(texts), batch_size):
        batch = mt_tok(texts[i:i + batch_size], return_tensors="pt", padding=True,
                       truncation=True, max_length=256).to(DEVICE)
        with torch.no_grad():
            gen = mt_model.generate(**batch, max_new_tokens=MT_MAX_NEW_TOKENS, num_beams=NUM_BEAMS)
        out += mt_tok.batch_decode(gen, skip_special_tokens=True)
    return out


t0 = time.time()
hypotheses = translate(sources)
print(f"Translated {len(hypotheses)} sentences in {time.time() - t0:.1f} s")
print("EN :", sources[0]); print("MT :", hypotheses[0]); print("REF:", references[0])

In [ ]:
import sacrebleu

bleu = sacrebleu.corpus_bleu(hypotheses, [references])   # [references]: a list of reference sets
chrf = sacrebleu.corpus_chrf(hypotheses, [references])
print(bleu)
print(chrf)
print("n-gram precisions (%):", [round(p, 1) for p in bleu.precisions])
print(f"brevity penalty: {bleu.bp:.3f} (system length {bleu.sys_len}, reference length {bleu.ref_len})")

# The signature records the exact settings (tokenizer, casing, smoothing) so others can reproduce a score.
bleu_metric = sacrebleu.BLEU()
bleu_metric.corpus_score(hypotheses, [references])
print("BLEU signature:", bleu_metric.get_signature())

`corpus_bleu` pools the n-gram counts of **all** sentences before computing the score; it is not
the average of sentence-level BLEU. Sentence-level scores are noisy but useful to *find* bad
cases. Let us look at the sentences with the lowest BLEU.

In [ ]:
per_sentence = pd.DataFrame({
    "source (EN)": sources,
    "model (ES)": hypotheses,
    "reference (ES)": references,
    "BLEU": [sacrebleu.sentence_bleu(h, [r]).score for h, r in zip(hypotheses, references)],
    "chrF": [sacrebleu.sentence_chrf(h, [r]).score for h, r in zip(hypotheses, references)],
}).round(1)
display(per_sentence.sort_values("BLEU").head(3))

Read those rows carefully. Is the model's translation really bad, or is the reference a *free*,
literary translation (the translator reordered, merged or added words)? The references in
`opus_books` come from published book translations, and the sentence alignment is automatic, so
some pairs are loose. A few "English" rows are not even in English: with the default sample of 60
you may find a sentence in another language (e.g., French) among the lowest scores.
A low score can be the reference's fault: read the data before blaming the model.

**Contamination caveat.** OPUS-MT models are trained on data from the OPUS collection, and
`opus_books` is part of OPUS. Some of these sentences may have been seen during training, which
would make the scores optimistic. Always ask: *could the model have seen my test set?*

### What BLEU and chrF reward (and what they miss)

One reference, four candidates. Predict the ranking before running the cell.

In [ ]:
ref = "El banco aprobó ayer la transferencia del cliente."
candidates = {
    "identical": "El banco aprobó ayer la transferencia del cliente.",
    "paraphrase": "Ayer, la transferencia del cliente fue aprobada por el banco.",
    "negated (wrong meaning)": "El banco no aprobó ayer la transferencia del cliente.",
    "too short": "El banco aprobó.",
}
display(pd.DataFrame({
    name: {"BLEU": sacrebleu.sentence_bleu(c, [ref]).score, "chrF": sacrebleu.sentence_chrf(c, [ref]).score}
    for name, c in candidates.items()
}).T.round(1))

The negated sentence means the opposite, yet it scores far above the correct paraphrase. The
"too short" candidate has perfect unigram precision, but the brevity penalty pulls its BLEU down.
N-gram overlap measures *surface similarity to one reference*, not correctness.

✋ **Checkpoint:** Our corpus BLEU is on a 0–100 scale. Is it "good"? Compared with what? (Hint:
a score only means something next to a baseline, on the same test set, with the same signature.)

🧪 **Try it:** build a trivial baseline that "translates" by copying the English source, and score it.

<details><summary>Show a solution</summary>

```python
copy_bleu = sacrebleu.corpus_bleu(sources, [references])
copy_chrf = sacrebleu.corpus_chrf(sources, [references])
print(f"copy-the-source baseline: BLEU {copy_bleu.score:.1f} · chrF {copy_chrf.score:.1f}")
```
The baseline is not zero (names, numbers and punctuation match), which shows how much of a
score is "free". The model should be far above it.
</details>

In [ ]:
del mt_model
release_memory()
print("Translation model released.")

## Part 3 — Summarization: ROUGE, a baseline and semantic similarity

We summarise news articles from `abisee/cnn_dailymail` (version 3.0.0) with `google/flan-t5-small`
and compare each summary with the human-written highlights. The dataset is large, so we **stream**
it: only the articles we use are downloaded.

**ROUGE** (Lin, 2004) is recall-oriented: how much of the reference is covered by the candidate.
ROUGE-1 and ROUGE-2 count overlapping unigrams and bigrams; ROUGE-L uses the longest common
subsequence (LCS), which rewards words in the same order without requiring them to be contiguous.
We report the F1 version, as the `rouge_score` package does.

In [ ]:
news_stream = load_dataset("abisee/cnn_dailymail", "3.0.0", split="validation", streaming=True)
articles, highlights = [], []
for example in news_stream:
    articles.append(example["article"])
    highlights.append(example["highlights"])
    if len(articles) == N_SUMMARIES:
        break
print(f"{len(articles)} articles · first one has {len(articles[0].split())} words")
print("Reference highlights:\n", highlights[0])

In [ ]:
SUM_NAME = "google/flan-t5-small"
sum_tok = AutoTokenizer.from_pretrained(SUM_NAME)
sum_model = AutoModelForSeq2SeqLM.from_pretrained(SUM_NAME).to(DEVICE).eval()


def summarize(article, prefix="summarize: "):
    inputs = sum_tok(prefix + article, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        out = sum_model.generate(**inputs, max_new_tokens=SUM_MAX_NEW_TOKENS, num_beams=NUM_BEAMS)
    return sum_tok.decode(out[0], skip_special_tokens=True)


def lead3(article):
    """Baseline: the first three sentences of the article."""
    sentences = re.split(r"(?<=[.!?])\s+", article.strip())
    return " ".join(sentences[:3])


t0 = time.time()
model_summaries = [summarize(a) for a in articles]
baseline_summaries = [lead3(a) for a in articles]
print(f"{len(model_summaries)} summaries in {time.time() - t0:.1f} s")
print("MODEL :", model_summaries[0])
print("LEAD-3:", baseline_summaries[0][:300], "…")

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)


def rouge_f1(candidates, refs):
    rows = [scorer.score(r, c) for c, r in zip(candidates, refs)]   # score(target, prediction)
    return pd.DataFrame([{k: v.fmeasure for k, v in row.items()} for row in rows])


rouge_model = rouge_f1(model_summaries, highlights)
rouge_lead3 = rouge_f1(baseline_summaries, highlights)
rouge_table = pd.DataFrame({"flan-t5-small": rouge_model.mean(), "lead-3 baseline": rouge_lead3.mean()}).T
display(rouge_table.round(3))

**Why a baseline?** CNN/DailyMail highlights often reuse sentences from the top of the article,
so "copy the first three sentences" (lead-3) is a notoriously strong baseline for this dataset.
A model that does not clearly beat lead-3 is not obviously useful, and a model that beats it
may still invent facts. ROUGE also rewards length: lead-3 summaries are longer than the model's.
With 12 articles, differences of a point or two are within the noise: look at the per-article table.

**Contamination caveat.** FLAN-T5 was instruction-tuned on a large mixture of tasks that includes
summarisation datasets; it may have seen CNN/DailyMail *training* articles. We use the validation
split, but the style is familiar to the model.

### Semantic similarity with sentence embeddings

N-gram metrics miss paraphrases. An embedding model maps each text to a vector; the **cosine
similarity** of two vectors is high when the texts talk about the same thing, even with different
words. We use `sentence-transformers/all-MiniLM-L6-v2` (English). BERTScore (Zhang et al., 2020)
refines the idea by matching *token* embeddings between candidate and reference.

In [ ]:
from sentence_transformers import SentenceTransformer, util

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)


def cosine(cands, refs):
    a = embedder.encode(cands, convert_to_tensor=True, normalize_embeddings=True)
    b = embedder.encode(refs, convert_to_tensor=True, normalize_embeddings=True)
    return util.cos_sim(a, b).diagonal().cpu().numpy()


summary_view = pd.DataFrame({
    "model summary": model_summaries,
    "ROUGE-L model": rouge_model["rougeL"],
    "ROUGE-L lead-3": rouge_lead3["rougeL"],
    "cosine model": cosine(model_summaries, highlights),
    "cosine lead-3": cosine(baseline_summaries, highlights),
}).round(3)
display(summary_view)
print("Means:", summary_view.drop(columns="model summary").mean().round(3).to_dict())

Now three hand-made pairs that expose the limits of both families of metrics.

In [ ]:
pairs = {
    "paraphrase": ("The customer asked to cancel her credit card.", "The client wants to close her card account."),
    "negation": ("The transfer to the supplier was approved.", "The transfer to the supplier was not approved."),
    "unrelated": ("The transfer to the supplier was approved.", "Heavy rain is expected in Bogotá this weekend."),
}
display(pd.DataFrame({
    name: {"ROUGE-L F1": scorer.score(a, b)["rougeL"].fmeasure, "cosine": float(cosine([b], [a])[0])}
    for name, (a, b) in pairs.items()
}).T.round(3))

The negated sentence scores high on **both** metrics: neither checks meaning in detail. This is why
generation is also evaluated by people (fluency, faithfulness, usefulness) and, increasingly, by an
**LLM-as-a-judge** (Zheng et al., 2023). A judge is cheap and fast, but it has known biases:
*position* (prefers the first or second answer), *verbosity* (prefers longer answers) and
*self-enhancement* (prefers answers written by itself). Mitigate them: swap the order and average,
give a rubric, and check the judge against a sample of human ratings.

A judge prompt you can paste into any assistant:

```text
You are evaluating two summaries of the same news article.
Criteria: (1) faithfulness: no facts that are not in the article; (2) coverage of the key points;
(3) concision. Article: <ARTICLE> · Summary A: <A> · Summary B: <B>
Answer with "A", "B" or "tie", then one sentence of justification per criterion.
```

✋ **Checkpoint:** Did flan-t5-small beat lead-3 on ROUGE-L? On cosine similarity? Would you ship it
for Andes Bank's internal news digest based on these numbers alone?

🧪 **Try it:** use the judge prompt above with one article, the model summary and the lead-3 summary.
Run it twice, swapping A and B. Does the verdict change?

<details><summary>Show a solution</summary>

```python
i = 0
prompt = (f"You are evaluating two summaries of the same news article. ... Article: {articles[i][:2000]}"
          f" · Summary A: {model_summaries[i]} · Summary B: {baseline_summaries[i]}")
print(prompt)   # paste it into your assistant, then swap A and B and paste again
```
If the verdict flips when you swap the order, you have observed position bias: report the judge's
result only when both orders agree, or average over both orders.
</details>

In [ ]:
del sum_model
release_memory()
print("Summarization model released.")

## Part 4 — Classification: stratified k-fold, confusion matrix and error analysis

Many "generative" products contain a classifier (routing, intent detection, fraud flags). We use
10 intents of `legacy-datasets/banking77` (online-banking customer messages) and a transparent
model: TF-IDF features + logistic regression.

The protocol from the slides:
1. Keep the official **test** split untouched until the very end.
2. On the training split, estimate performance with **stratified k-fold cross-validation**
   (every fold keeps the class proportions). Report the mean **and** the spread.
3. Put preprocessing *inside* a `Pipeline`, so the TF-IDF vocabulary is learned only from the
   training folds (otherwise information from the validation fold leaks into training).
4. Evaluate the final model **once** on the test split, then analyse its errors.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score

INTENTS = ["card_arrival", "card_delivery_estimate", "lost_or_stolen_card", "card_payment_not_recognised",
           "pending_transfer", "transfer_not_received_by_recipient", "exchange_rate",
           "card_payment_wrong_exchange_rate", "pending_top_up", "top_up_failed"]

banking = load_dataset("legacy-datasets/banking77")
names = banking["train"].features["label"].names
keep = {names.index(i) for i in INTENTS}


def to_frame(split):
    df = split.to_pandas()
    df = df[df["label"].isin(keep)].copy()
    df["intent"] = df["label"].map(lambda i: names[i])
    return df[["text", "intent"]].reset_index(drop=True)


train_df, test_df = to_frame(banking["train"]), to_frame(banking["test"])
if N_PER_INTENT:
    train_df = train_df.groupby("intent", group_keys=False).sample(N_PER_INTENT, random_state=SEED)
print(f"train: {len(train_df)} · test: {len(test_df)}")
display(train_df["intent"].value_counts().rename("training examples").to_frame().T)


def make_model():
    return make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True),
                         LogisticRegression(max_iter=2000))

In [ ]:
K_FOLDS = 5
cv = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
cv_res = cross_validate(make_model(), train_df["text"], train_df["intent"], cv=cv,
                        scoring=["accuracy", "f1_macro"])
folds = pd.DataFrame({"accuracy": cv_res["test_accuracy"], "macro-F1": cv_res["test_f1_macro"]},
                     index=[f"fold {i + 1}" for i in range(K_FOLDS)])
display(folds.round(3))
print(f"{K_FOLDS}-fold macro-F1: {folds['macro-F1'].mean():.3f} ± {folds['macro-F1'].std():.3f} (mean ± std)")

**How much does a single hold-out split move?** Below we repeat a random 80/20 split several
times. Each repetition is what you would report if you had made *only one* split.

In [ ]:
holdout = []
for r in range(N_HOLDOUT_REPEATS):
    X_tr, X_va, y_tr, y_va = train_test_split(train_df["text"], train_df["intent"], test_size=0.2,
                                              stratify=train_df["intent"], random_state=r)
    holdout.append(f1_score(y_va, make_model().fit(X_tr, y_tr).predict(X_va), average="macro"))
holdout = np.array(holdout)
print(f"{N_HOLDOUT_REPEATS} single 80/20 splits: macro-F1 from {holdout.min():.3f} to {holdout.max():.3f}")

Now we train on the whole training split and evaluate **once** on the untouched test split.

In [ ]:
final_model = make_model().fit(train_df["text"], train_df["intent"])
test_pred = final_model.predict(test_df["text"])
print(classification_report(test_df["intent"], test_pred, digits=3, zero_division=0))

fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(test_df["intent"], test_pred, labels=INTENTS,
                                        xticks_rotation=90, colorbar=False, ax=ax)
ax.set_title("Confusion matrix on the test split")
plt.tight_layout(); plt.show()

### Error analysis: which intents does the model confuse?

The off-diagonal cells of the confusion matrix are the errors. We list the most frequent
(true → predicted) pairs and read real examples: that is where improvement ideas come from.

In [ ]:
cm = confusion_matrix(test_df["intent"], test_pred, labels=INTENTS)
pairs_cm = [(INTENTS[i], INTENTS[j], int(cm[i, j]))
            for i in range(len(INTENTS)) for j in range(len(INTENTS)) if i != j and cm[i, j] > 0]
top_pairs = pd.DataFrame(pairs_cm, columns=["true intent", "predicted as", "count"]) \
              .sort_values("count", ascending=False).reset_index(drop=True)
display(top_pairs.head(5))

if len(top_pairs):
    true_i, pred_i = top_pairs.loc[0, "true intent"], top_pairs.loc[0, "predicted as"]
    errors = test_df[(test_df["intent"] == true_i) & (test_pred == pred_i)]
    print(f"Messages labelled '{true_i}' but predicted as '{pred_i}':")
    for text in errors["text"].head(5):
        print("  •", text)

**Error analysis workflow** (the one you will use in the final project):
1. Sample the errors (here: the top confused pair).
2. Group them into categories (ambiguous message, overlapping label definitions, missing
   vocabulary, typo, label noise…) and count each category.
3. Write one hypothesis per category and one change that would fix it.
4. Try the change, compare with cross-validation on the **training** data, and only at the end
   confirm on the test split.

✋ **Checkpoint:**
1. Is the spread between folds (± std) larger or smaller than the range of the single 80/20 splits? What does that tell you about reporting one number?
2. For the top confused pair, is it a model error or a label-definition problem? Would a person always agree with the labels?

🧪 **Try it:** add character n-grams to the features and compare **with cross-validation on the
training split** (not with the test split).

<details><summary>Show a solution</summary>

```python
from sklearn.pipeline import make_union
def make_model_chars():
    feats = make_union(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True),
                       TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True))
    return make_pipeline(feats, LogisticRegression(max_iter=2000))
res = cross_validate(make_model_chars(), train_df["text"], train_df["intent"], cv=cv, scoring="f1_macro")
print(f"with char n-grams: {res['test_score'].mean():.3f} ± {res['test_score'].std():.3f}")
```
Character n-grams help with typos and word variants. Decide with the cross-validation numbers; if
you chose by looking at the test split, the test score would no longer be an unbiased estimate.
</details>

## Wrap-up

- **Perplexity** measures how well a model predicts a text; compare it only on the same text and with the same tokenizer.
- **BLEU** is precision-based with a brevity penalty; **ROUGE** is recall-oriented; **chrF** works on characters. All three measure overlap with a reference, not correctness.
- **Embedding similarity** captures paraphrases but still misses negations and invented facts: keep people (or a carefully checked LLM judge) in the loop.
- Always report a **baseline** (copy-the-source, lead-3, majority class) and the **spread** of your estimate (k-fold mean ± std).
- Keep a **test set** that you touch once; cross-validation is for choosing, the test set is for confirming.

➡️ **Now the Session Challenge:** you will compute these metrics on *your own* sample of news and
sentences, find a case where the metric disagrees with your judgment, and design an evaluation
plan for a business product.